# Group 2 — Application & Integration

**Application and Integration Lead:** Nze Chime

## Purpose

This notebook prepares, validates, and launches the Group 2 Telecom Customer Churn prototype from Google Colab.

The complete `telecom_churn_app` folder must be uploaded directly into `/content`. The notebook does not depend on Google Drive.

The workflow covers environment preparation, application verification, model-artifact verification, inference testing, Streamlit startup, automatic free-port selection, server health checking, and browser access.


## 1. Install the application environment

The saved preprocessing pipeline and trained model require the compatible versions of scikit-learn, pandas, numpy, and joblib shown below. The versions are pinned to keep the serialized artifacts compatible.

Streamlit is installed as the framework used to serve the prototype.


In [1]:
# Install Streamlit and the exact machine-learning library versions required by the saved artifacts.
!pip install -q streamlit==1.50.0 pandas==2.2.3 scikit-learn==1.6.1 numpy==2.1.3 joblib==1.5.3

# Confirm that the installation completed.
print("Application environment installed successfully.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 3.6 MB/s eta 0:00:00
Application environment installed successfully.


## 2. Import the required libraries

All Python imports are kept in one cell. The imports support filesystem management, model validation, background process execution, timing, HTTP health checks, and Colab browser access.


In [2]:
# Import os for working with the Colab runtime environment.
import os

# Import subprocess for starting Streamlit as a background process.
import subprocess

# Import time for startup delays and health-check retry intervals.
import time

# Import warnings for suppressing non-critical warnings.
import warnings

# Import Path for reliable filesystem path handling.
from pathlib import Path

# Import urlopen for testing the Streamlit HTTP endpoint.
from urllib.request import urlopen

# Import pandas for the controlled model inference test.
import pandas as pd

# Import joblib for loading the saved model artifacts.
import joblib

# Import scikit-learn so the active version can be displayed.
import sklearn

# Import NumPy so the active version can be displayed.
import numpy

# Import the Colab output interface for exposing the Streamlit port.
from google.colab import output

# Suppress non-critical warnings to keep the notebook output readable.
warnings.filterwarnings("ignore")

# Display the active scikit-learn version.
print("scikit-learn:", sklearn.__version__)

# Display the active pandas version.
print("pandas:", pd.__version__)

# Display the active NumPy version.
print("numpy:", numpy.__version__)

# Display the active joblib version.
print("joblib:", joblib.__version__)


scikit-learn: 1.6.1
pandas: 2.2.3
numpy: 2.1.3
joblib: 1.5.3


In [3]:
# Define the uploaded application directory.
APP_DIR = Path("/content/telecom_churn_app")

# Define the Streamlit application entrypoint.
APP_FILE = APP_DIR / "app.py"

# Define the directory containing the machine-learning artifacts.
MODEL_DIR = APP_DIR / "models"

# Define the saved preprocessing pipeline.
PREPROCESSOR_FILE = MODEL_DIR / "preprocessor.joblib"

# Define the saved tuned Random Forest model.
MODEL_FILE = MODEL_DIR / "best_random_forest_tuned.pkl"

# Define the optional seed-customer dataset.
SEED_FILE = APP_DIR / "prototype_seed_customers.csv"

# Display the application directory being used.
print("Application directory:", APP_DIR)


Application directory: /content/telecom_churn_app


### Application Path Configuration

Defines the locations of the Streamlit app, model artifacts, and customer seed data within the uploaded project folder. This keeps all file references centralized and ensures the application loads its required components from the correct paths.

## 3. Verify the uploaded application package

The application should not be launched until its entrypoint and model artifacts have been confirmed. This validation gate catches missing or incorrectly placed files early.


In [4]:
# Define the files that are required for model-backed predictions.
required_files = [
    APP_FILE,
    PREPROCESSOR_FILE,
    MODEL_FILE,
]

# Check each required file.
for required_file in required_files:
    # Stop execution if a required file is missing.
    if not required_file.exists():
        raise FileNotFoundError(f"Required application file not found: {required_file}")

    # Report that the current file exists.
    print("OK:", required_file)

# Check whether the seed dataset is present.
if SEED_FILE.exists():
    # Report that the seed dataset is available.
    print("OK:", SEED_FILE)
else:
    # Report that the optional seed dataset is not present.
    print("INFO: prototype_seed_customers.csv was not found.")

# Confirm that the package validation completed.
print("Application package verification completed.")


OK: /content/telecom_churn_app/app.py
OK: /content/telecom_churn_app/models/preprocessor.joblib
OK: /content/telecom_churn_app/models/best_random_forest_tuned.pkl
OK: /content/telecom_churn_app/prototype_seed_customers.csv
Application package verification completed.


## 5. Inspect the application structure

The uploaded directory is displayed recursively so the We can confirm that the application has been uploaded at the correct level and that the model artifacts are inside the expected `models` directory.


In [5]:
# Display every directory and file under the application root.
for path in sorted(APP_DIR.rglob("*")):
    # Display directories with a directory marker.
    if path.is_dir():
        print(f"[DIR ] {path.relative_to(APP_DIR)}/")
    else:
        # Display files together with their sizes.
        print(f"[FILE] {path.relative_to(APP_DIR)} — {path.stat().st_size:,} bytes")

# Confirm that the structure inspection completed.
print("Application structure inspection completed.")


[DIR ] .ipynb_checkpoints/
[FILE] app.py — 40,127 bytes
[DIR ] models/
[FILE] models/best_random_forest_tuned.pkl — 2,619,193 bytes
[FILE] models/preprocessor.joblib — 7,735 bytes
[FILE] prototype_seed_customers.csv — 10,295 bytes
[FILE] requirements.txt — 80 bytes
Application structure inspection completed.


## 6. Load and validate the saved model artifacts

The preprocessing pipeline and tuned Random Forest are loaded independently of Streamlit. This separates model compatibility problems from application-serving problems.


In [6]:
# Load the saved preprocessing pipeline.
preprocessor = joblib.load(PREPROCESSOR_FILE)

# Load the saved tuned Random Forest model.
model = joblib.load(MODEL_FILE)

# Display the preprocessing pipeline type.
print("Preprocessor:", type(preprocessor))

# Display the model type.
print("Model:", type(model))

# Confirm that the model provides class prediction.
print("predict available:", hasattr(model, "predict"))

# Confirm that the model provides probability prediction.
print("predict_proba available:", hasattr(model, "predict_proba"))

# Confirm that artifact loading completed successfully.
print("Model artifact validation completed.")


Preprocessor: <class 'sklearn.compose._column_transformer.ColumnTransformer'>
Model: <class 'sklearn.ensemble._forest.RandomForestClassifier'>
predict available: True
predict_proba available: True
Model artifact validation completed.


Technical significance: Confirms that the exact artifacts required for preprocessing, classification, and probability-based churn scoring are available before the application proceeds to inference.

## 7. Run a controlled inference test

One representative customer record is passed through the saved preprocessing pipeline and model. This confirms that the artifacts can perform inference before the web application is launched.


In [7]:
# Create one representative customer record using the model's expected feature names.
test_customer = pd.DataFrame([{
    # Set tenure in months.
    "tenure": 12,
    # Set monthly charges.
    "MonthlyCharges": 65.0,
    # Set cumulative charges.
    "TotalCharges": 780.0,
    # Set gender.
    "gender": "Female",
    # Set senior-citizen indicator.
    "SeniorCitizen": "0",
    # Set partner status.
    "Partner": "No",
    # Set dependent status.
    "Dependents": "No",
    # Set phone-service status.
    "PhoneService": "Yes",
    # Set multiple-line status.
    "MultipleLines": "No",
    # Set internet-service type.
    "InternetService": "Fiber optic",
    # Set online-security status.
    "OnlineSecurity": "No",
    # Set online-backup status.
    "OnlineBackup": "No",
    # Set device-protection status.
    "DeviceProtection": "No",
    # Set technical-support status.
    "TechSupport": "No",
    # Set streaming-TV status.
    "StreamingTV": "Yes",
    # Set streaming-movies status.
    "StreamingMovies": "Yes",
    # Set contract type.
    "Contract": "Month-to-month",
    # Set paperless-billing status.
    "PaperlessBilling": "Yes",
    # Set payment method.
    "PaymentMethod": "Electronic check",
}])

# Transform the customer record using the saved preprocessing pipeline.
X_test = preprocessor.transform(test_customer)

# Generate the model class prediction.
prediction = model.predict(X_test)[0]

# Generate probabilities for both classes.
probabilities = model.predict_proba(X_test)[0]

# Extract the retention probability.
retain_probability = probabilities[0]

# Extract the churn probability.
churn_probability = probabilities[1]

# Define the team's locked churn threshold.
threshold = 0.40

# Apply the locked threshold to the churn probability.
threshold_prediction = int(churn_probability >= threshold)

# Display the model's class prediction.
print("Model prediction:", prediction)

# Display the retention probability.
print(f"Retention probability: {retain_probability * 100:.1f}%")

# Display the churn probability.
print(f"Churn probability: {churn_probability * 100:.1f}%")

# Display the business-facing prediction produced by the 40% threshold.
print(f"40% threshold prediction: {threshold_prediction}")


Model prediction: 1
Retention probability: 25.5%
Churn probability: 74.5%
40% threshold prediction: 1


### Outcome

The inference test should return a prediction and probabilities. The final value demonstrates how the locked 40% threshold converts churn probability into the application's business-facing churn flag.


## 8. Launch Streamlit on an automatically selected free port

The application is started as a background process so the notebook remains responsive.

The notebook searches ports 8501–8510 and automatically selects the first available one. This means a previous Streamlit process occupying one port does not require the notebook to be manually edited.

For Colab's proxy environment, CORS, XSRF protection, and WebSocket compression are disabled for this local prototype launch. These settings are specifically for the Colab integration environment.


In [8]:
# Change the working directory to the uploaded application.
os.chdir(APP_DIR)

# Define the first port to test.
FIRST_PORT = 8501

# Define the last port to test.
LAST_PORT = 8510

# Define a helper function for checking whether a port is available.
def port_is_available(port):
    # Import the socket library for the port test.
    import socket

    # Create a TCP socket.
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

    # Attempt to bind the socket to the candidate port.
    try:
        sock.bind(("127.0.0.1", port))

        # Return True when binding succeeds.
        return True

    # Always close the socket after the test.
    finally:
        sock.close()

# Initialise the selected port as empty.
STREAMLIT_PORT = None

# Test each port in the configured range.
for candidate_port in range(FIRST_PORT, LAST_PORT + 1):
    # Select the first available port.
    if port_is_available(candidate_port):
        STREAMLIT_PORT = candidate_port
        break

# Stop if no free port was found.
if STREAMLIT_PORT is None:
    raise RuntimeError("No available Streamlit port was found from 8501 to 8510.")

# Define the Streamlit log file.
STREAMLIT_LOG = Path("/content/streamlit.log")

# Remove the previous log file if it exists.
if STREAMLIT_LOG.exists():
    STREAMLIT_LOG.unlink()

# Build the Streamlit launch command.
streamlit_command = [
    "streamlit",
    "run",
    "app.py",
    "--server.port",
    str(STREAMLIT_PORT),
    "--server.address",
    "0.0.0.0",
    "--server.headless",
    "true",
    "--server.fileWatcherType",
    "none",
    "--server.enableCORS",
    "false",
    "--server.enableXsrfProtection",
    "false",
    "--server.enableWebsocketCompression",
    "false",
]

# Open the log file for Streamlit output.
log_handle = open(STREAMLIT_LOG, "w", encoding="utf-8")

# Start Streamlit as a background process.
streamlit_process = subprocess.Popen(
    streamlit_command,
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    text=True,
)

# Wait briefly for Streamlit to initialise.
time.sleep(5)

# Check whether Streamlit exited during startup.
if streamlit_process.poll() is not None:
    # Close the log file.
    log_handle.close()

    # Display the Streamlit startup log.
    print(STREAMLIT_LOG.read_text(encoding="utf-8"))

    # Stop because the application did not remain running.
    raise RuntimeError("Streamlit failed during startup.")

# Display the selected port and process ID.
print("Streamlit started successfully.")
print("Port:", STREAMLIT_PORT)
print("PID:", streamlit_process.pid)
print("Log:", STREAMLIT_LOG)


Streamlit started successfully.
Port: 8501
PID: 7625
Log: /content/streamlit.log


### Outcome

The cell should report a Streamlit process ID and the automatically selected port. The process runs in the background, allowing the notebook to continue to the server-readiness check.


## 9. Verify the Streamlit server before opening the browser

A running process is not enough to prove that the web application is ready. The Streamlit health endpoint is checked repeatedly before the public Colab URL is generated.

If Streamlit exits or never becomes reachable, the notebook displays `/content/streamlit.log` so the failure can be diagnosed.


In [9]:
# Define the local Streamlit health endpoint.
health_url = f"http://127.0.0.1:{STREAMLIT_PORT}/_stcore/health"

# Define the maximum number of health-check attempts.
MAX_ATTEMPTS = 30

# Track whether the server becomes ready.
server_ready = False

# Check the health endpoint repeatedly while Streamlit starts.
for attempt in range(1, MAX_ATTEMPTS + 1):
    # Check whether the Streamlit process has exited.
    if streamlit_process.poll() is not None:
        # Close the log file.
        log_handle.close()

        # Display the Streamlit log.
        print(STREAMLIT_LOG.read_text(encoding="utf-8"))

        # Stop because the server exited unexpectedly.
        raise RuntimeError("Streamlit exited before becoming ready.")

    # Attempt to contact the Streamlit health endpoint.
    try:
        # Open the health endpoint with a short timeout.
        with urlopen(health_url, timeout=2) as response:
            # Mark the server as ready when HTTP 200 is returned.
            if response.status == 200:
                server_ready = True
                break

    # Ignore temporary connection failures while the server starts.
    except Exception:
        # Wait briefly before trying again.
        time.sleep(1)

# Close the Streamlit log after the startup check.
log_handle.close()

# Stop if the server never became ready.
if not server_ready:
    # Display the final Streamlit log.
    print(STREAMLIT_LOG.read_text(encoding="utf-8"))

    # Stop execution with a clear error.
    raise RuntimeError("Streamlit did not become reachable within the startup window.")

# Confirm that Streamlit is responding.
print("Streamlit health check: PASSED")

# Display the local health endpoint.
print("Local health endpoint:", health_url)


Streamlit health check: PASSED
Local health endpoint: http://127.0.0.1:8501/_stcore/health


### Outcome

The expected result is `Streamlit health check: PASSED`. This confirms that Streamlit is actually serving requests before the browser link is exposed.


## 10. Generate the Colab browser link

The verified Streamlit port is exposed through the Google Colab browser proxy. A fresh URL is generated from the automatically selected port.


In [10]:
# Generate a browser-accessible URL for the verified Streamlit port.
streamlit_url = output.eval_js(
    f"google.colab.kernel.proxyPort({STREAMLIT_PORT})"
)

# Display the application-ready heading.
print("==============================================")

# Display the application-ready status.
print("STREAMLIT APPLICATION IS READY")

# Display the closing separator.
print("==============================================")

# Display the selected port.
print("Port:", STREAMLIT_PORT)

# Display the browser-accessible application URL.
print("Open the prototype here:")

# Display the generated Colab proxy URL.
print(streamlit_url)


STREAMLIT APPLICATION IS READY
Port: 8501
Open the prototype here:
https://8501-m-s-kkb-usc1b1-1weuupsy4i5oe-b.us-central1-1.prod.colab.dev


### Application integration completion criteria

The prototype is ready when this cell displays `STREAMLIT APPLICATION IS READY` followed by the generated Colab URL.

## Integration completion criteria

- The complete application folder is present under `/content/telecom_churn_app`.
- The saved preprocessing pipeline loads successfully.
- The tuned Random Forest loads successfully.
- Controlled inference succeeds.
- The 40% churn threshold is applied.
- Streamlit starts as a background process.
- An available port is selected automatically.
- The Streamlit health endpoint returns HTTP 200.
- The Colab browser URL is generated only after the health check passes.

If the application does not start, inspect `/content/streamlit.log`. The notebook is designed to expose the server log when startup or health checking fails.
